# Train YOLOv8 on Indian Traffic Data (Colab free T4) — Official BMD-45 run

Purpose: train a YOLOv8 detector on the **official `iisc-aim/BMD-45`** release
(CC-BY-4.0, commercial OK; CVPR 2026 Findings) with the **Option B merge**
(14 raw classes → 6 contract ids), evaluated on the **official val split**
(10,194 images — no synthetic split, numbers comparable to the paper).

## Why chunked (153 GB source, ~100 GB disk)
Official train+val ships as PNGs totaling ~153 GB. The converter downloads
**one image folder at a time**, transcodes PNG→JPG (q92) into the contract
set (~18 GB total, peak ~35 GB), deletes PNGs, and continues. Box coords are
normalized in COCO space, so the transcode changes no label math.
Trim `TRAIN_FOLDERS` in setup for a shorter pilot run.

## Data flow (only the registry zip leaves Colab)
1. Annotation JSONs first (KBs) → full label tables before pixels move.
2. Chunked pixels → contract JPGs + YOLO labels → official val as val.
3. Train (30ep/batch32) -> val -> ONNX -> static int8 -> registry zip.

## Auth for speed
Colab Secrets key **`HF_TOKEN`** → higher HF rate limits; picked up
automatically, anonymous otherwise. Pair with higher `max_workers` if slow.

## Runtime instructions
1. `Runtime` -> **T4 GPU**. Run code cells one at a time, confirming output.
2. Prep (download+transcode) takes ~1h, training ~4-5h: keep tab focused.


## Setup

Installs deps, selects folders, checks ~45 GB free disk, and pulls the HF client. No pixels move here — the converter cell downloads chunk by chunk.

In [ ]:
!pip install -q ultralytics onnx onnxruntime huggingface_hub

import os

# ---- dataset source: official IISc AIM release (CC-BY-4.0, commercial OK) ----
HF_REPO = "iisc-aim/BMD-45"
HF_TRAIN = "BMD-45-Train"  # 35,792 imgs, images_000/..007 + _annotations.coco.json
HF_VAL = "BMD-45-Val"      # 10,194 imgs, images_000/..002 + _annotations.coco.json
# Trim TRAIN_FOLDERS for a shorter pilot (e.g. ["images_000"] ~ 4.5k images).
TRAIN_FOLDERS = ["images_%03d" % i for i in range(8)]
VAL_FOLDERS = ["images_%03d" % i for i in range(3)]
DRIVE_DATASET_ROOT = "/tmp/india_yolo_dataset"  # contract dataset (ephemeral)

# Auth (optional but recommended): Colab Secrets -> key name HF_TOKEN.
# Higher rate limits = fewer 429s. Absent? Anonymous fallback (slower).
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        print("HF auth: token loaded from Colab Secrets")
    else:
        print("HF auth: no HF_TOKEN secret, anonymous (slower)")
except Exception as e:
    print("HF auth: userdata unavailable (%s), anonymous" % e)

import shutil
free_gb = shutil.disk_usage("/tmp").free / 1e9
print("/tmp free: %.1f GB" % free_gb)
# peak = one PNG folder (~15 GB) + all JPGs (~18 GB) + torch/runs overhead
assert free_gb > 45, "need ~45 GB free for the chunked pipeline; free space and retry"
from huggingface_hub import snapshot_download
print("repo:", HF_REPO)
print("contract root:", DRIVE_DATASET_ROOT)


## Convert official BMD-45 to contract (in Colab)

Chunked PNG→JPG transcode + Option B merge + official val. Expect ~1h for
the full 8+3 folders (progress prints per folder). For a pilot, trim
`TRAIN_FOLDERS` in setup and re-run this cell.

In [ ]:
import os, glob, json, random, shutil
import cv2
from pathlib import Path

# Mirrors scripts/prepare_dataset.py::convert_coco (Option B) but chunked:
# ONE image folder at a time is downloaded, transcoded PNG->JPG (q92), then
# PNGs are deleted before the next folder. Peak disk stays ~35 GB for the
# full 153 GB source. Official BMD-45-Val is val (no synthetic split).
CLASSES = ["car", "motorcycle", "bus", "truck", "bicycle", "auto"]

ALIASES = {
    "motorbike": "motorcycle", "moto": "motorcycle",
    "two_wheeler": "motorcycle", "two-wheeler": "motorcycle",
    "bicycle": "bicycle", "bike": "bicycle", "cycle": "bicycle",
    "autorickshaw": "auto", "rickshaw": "auto",
    "three_wheeler": "auto", "auto_rickshaw": "auto",
    "three-wheeler": "auto",
}
# Option B merge: 14 BMD-45 classes -> 6 contract ids (Other is dropped).
MERGE_B = {
    "sedan": "car", "hatchback": "car", "suv": "car", "muv": "car",
    "minibus": "bus", "mini_bus": "bus", "van": "bus",
    "tempo_traveller": "bus", "tempo": "bus",
    "lcv": "truck",
}
ALIASES.update(MERGE_B)

def class_index(name):
    n = (name or "").lower().strip().replace(" ", "_").replace("-", "_")
    m = ALIASES.get(n, n)
    return CLASSES.index(m) if m in CLASSES else None

def load_coco(subset_root):
    base = Path(subset_root)
    afs = sorted(base.rglob("*annotations*.json"))
    assert afs, "no COCO JSON under " + str(subset_root)
    data = json.loads(afs[0].read_text(encoding="utf-8"))
    cats = {c["id"]: c["name"] for c in data.get("categories", [])}
    imgs = {im["id"]: im for im in data.get("images", [])}
    assert cats and imgs, "empty categories/images - wrong JSON file?"
    print(base.name + ": " + str(len(cats)) + " cats, " + str(len(imgs)) + " images, "
          + str(len(data.get("annotations", []))) + " annotations")
    return cats, imgs, data.get("annotations", [])

def convert_official_bmd(out_root, train_folders, val_folders):
    from huggingface_hub import snapshot_download
    # annotation JSONs first (KBs): full label tables before any pixels move
    meta = snapshot_download(repo_id=HF_REPO, repo_type="dataset",
                             allow_patterns=["BMD-45-Train/*.json", "BMD-45-Val/*.json"],
                             max_workers=2)
    t_cats, t_imgs, t_ann = load_coco(os.path.join(meta, HF_TRAIN))
    v_cats, v_imgs, v_ann = load_coco(os.path.join(meta, HF_VAL))
    assert t_cats.keys() == v_cats.keys(), "train/val category ids differ!"
    # per-image box lines, keyed by file stem (resolution-independent: normalized)
    jobs = {}
    kept_hist, dropped_hist = {}, {}
    for cats, imgs, anns, split in ((t_cats, t_imgs, t_ann, "train"),
                                    (v_cats, v_imgs, v_ann, "val")):
        for a in anns:
            cname = cats.get(a.get("category_id"))
            idx = class_index(cname) if cname else None
            if idx is None:
                dropped_hist[cname] = dropped_hist.get(cname, 0) + 1
                continue
            im = imgs.get(a.get("image_id"))
            if im is None:
                continue
            iw, ih = im.get("width") or 1, im.get("height") or 1
            x, y, w, h = a["bbox"]
            cx, cy = (x + w / 2) / iw, (y + h / 2) / ih
            stem = Path(str(im.get("file_name", ""))).stem
            kept_hist[CLASSES[idx]] = kept_hist.get(CLASSES[idx], 0) + 1
            line = str(idx) + " %.6f %.6f %.6f %.6f" % (cx, cy, w / iw, h / ih)
            jobs.setdefault((split, stem), []).append(line)
    print("kept per class:", kept_hist)
    print("dropped per raw class:", dropped_hist)
    assert jobs, "no kept boxes - check mapping"
    # chunked pixels: one folder at a time, transcode, delete, next
    counts = {"train": 0, "val": 0}
    n_missing = 0
    for subset, folders, split in ((HF_TRAIN, train_folders, "train"),
                                   (HF_VAL, val_folders, "val")):
        for fld in folders:
            d = snapshot_download(repo_id=HF_REPO, repo_type="dataset",
                                  allow_patterns=[subset + "/" + fld + "/**"],
                                  max_workers=2)
            src_dir = Path(d) / subset / fld
            for png in sorted(src_dir.glob("*.png")):
                lines = jobs.get((split, png.stem))
                if not lines:
                    n_missing += 1
                    continue
                img = cv2.imread(str(png))
                assert img is not None, "unreadable " + str(png)
                for sub in ("images", "labels"):
                    os.makedirs(os.path.join(out_root, sub, split), exist_ok=True)
                cv2.imwrite(os.path.join(out_root, "images", split, png.stem + ".jpg"),
                            img, [cv2.IMWRITE_JPEG_QUALITY, 92])
                fh = open(os.path.join(out_root, "labels", split, png.stem + ".txt"), "w")
                fh.write("\n".join(lines) + "\n")
                fh.close()
                counts[split] += 1
            shutil.rmtree(src_dir, ignore_errors=True)  # free peak disk before next folder
            print("done " + subset + "/" + fld + " -> " + str(counts))
    print("split images: " + str(counts) + "; w/o kept boxes: " + str(n_missing))
    assert counts["train"] > 0 and counts["val"] > 0, "empty split produced"
    names = ["path: " + os.path.abspath(out_root), "train: images/train",
             "val: images/val", "test: images/val", "names:"]
    names += ["  %d: %s" % (i, n) for i, n in enumerate(CLASSES)]
    os.makedirs(out_root, exist_ok=True)
    open(os.path.join(out_root, "data.yaml"), "w").write("\n".join(names) + "\n")
    print("data.yaml: official val used; test -> val (annotations withheld)")

# Guard against a stale kernel (jobs dict exists only in this function).
import inspect
assert "n_missing" in inspect.getsource(convert_official_bmd), "stale kernel: re-run this cell fully (or Restart and run all)"
convert_official_bmd(DRIVE_DATASET_ROOT, TRAIN_FOLDERS, VAL_FOLDERS)

# calibration frames: 200 random from the converted set.
rng = random.Random(42)
all_imgs = glob.glob(os.path.join(DRIVE_DATASET_ROOT, "images", "*", "*.jpg"))
calib_dir = os.path.join(DRIVE_DATASET_ROOT, "calibration", "frames")
os.makedirs(calib_dir, exist_ok=True)
for p in rng.sample(all_imgs, min(200, len(all_imgs))):
    shutil.copy2(p, os.path.join(calib_dir, os.path.basename(p)))
print("calibration: " + str(len(os.listdir(calib_dir))) + " frames from " + str(len(all_imgs)) + " images")


## OPTIONAL: persist the contract set to Drive (run once, after conversion)

The contract set in `/tmp` (~20-25 GB) is wiped on disconnect. This cell copies
images + labels + `data.yaml` to Drive so the NEXT session skips the ~1h
download+transcode entirely. Skips files already backed up (resume-safe).
Needs ~25 GB free on Drive — fails fast with real numbers if short.
Next session: mount Drive, copy back to `/tmp/india_yolo_dataset`, fix the
`path:` line in `data.yaml` to the new location, skip to Validate.

In [ ]:
import os, shutil

BACKUP_ROOT = "/content/drive/MyDrive/india_yolo_contract"  # persistent copy
SRC_ROOT = DRIVE_DATASET_ROOT  # /tmp/india_yolo_dataset (ephemeral)

from google.colab import drive
drive.mount("/content/drive")

def _dir_gb(p):
    tot = 0
    for root, _, files in os.walk(p):
        for f in files:
            try:
                tot += os.path.getsize(os.path.join(root, f))
            except OSError:
                pass
    return tot / 1e9

need = _dir_gb(SRC_ROOT)
free = shutil.disk_usage("/content/drive").free / 1e9
print("contract set: %.1f GB | Drive free: %.1f GB" % (need, free))
assert free > need + 1, "Drive too full: free %.1f GB, need %.1f GB" % (free, need)

n_new, n_skip = 0, 0
for root, _, files in os.walk(SRC_ROOT):
    for f in files:
        s = os.path.join(root, f)
        d = os.path.join(BACKUP_ROOT, os.path.relpath(s, SRC_ROOT))
        if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s):
            n_skip += 1
            continue
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copy2(s, d)
        n_new += 1
print("backed up: %s (%d new, %d already present)" % (BACKUP_ROOT, n_new, n_skip))
print("NEXT SESSION: mount Drive, copy %s -> /tmp/india_yolo_dataset, "
      "set data.yaml 'path:' to /tmp/india_yolo_dataset, run Validate." % BACKUP_ROOT)


## Validate dataset

Checks images-vs-labels counts per split, asserts exactly 6 classes from
`data.yaml`, warns on empty label files and out-of-range boxes. Self-contained;
no repo imports (Colab has no checkout). `test` points at `val` in this run
(single-JSON source has no held-out test) — the validator tolerates that.

## Train tier-low model: yolov8n @ 30 epochs (Option B)

BMD-45 is ~13x HeTra: 30 epochs @ batch 32 fits a free T4 session (~3-5h).
Class imbalance is extreme (two-wheeler >> bicycle) — defaults kept;
bicycle mAP is a watch-item, re-weight only on evidence.

In [ ]:
from ultralytics import YOLO

model_n = YOLO('yolov8n.pt')
model_n.train(
    data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'),
    epochs=30, imgsz=640, batch=32, patience=10, workers=4,
    # GPU OOM? batch=16. Host-RAM kill (runtime disconnect)? workers=2.
    name='india_yolov8n',
)

run_dir_n = model_n.trainer.save_dir
results_csv = os.path.join(run_dir_n, 'results.csv')
print('\n--- results.csv (head/tail snippet) ---')
with open(results_csv) as f:
    lines = f.readlines()
for ln in lines[:2] + lines[-5:]:
    print(ln.rstrip())
BEST_N = os.path.join(run_dir_n, 'weights', 'best.pt')
print('best.pt:', BEST_N)

In [ ]:
metrics_n = YOLO(BEST_N).val(data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'), verbose=False)

def print_map_table(metrics):
    # ultralytics Metric: .ap_class_index, .p, .r are arrays over present classes;
    # .ap50 is per-class AP@0.5; .map50 is overall scalar
    names = metrics.names  # {id: name}
    idx = metrics.box.ap_class_index
    header = f"{'class':<12} {'precision':>9} {'recall':>9} {'mAP50':>9}"
    print(header)
    print('-' * len(header))
    rows = {}
    for i, ci in enumerate(idx):
        name = names[int(ci)]
        p = float(metrics.box.p[i]); r = float(metrics.box.r[i])
        m = float(metrics.box.ap50[i])
        mark = '>>>' if name == 'auto' else '   '
        print(f"{mark} {name:<9} {p:>9.3f} {r:>9.3f} {m:>9.3f}")
        rows[name] = round(m, 4)
    print(f"\noverall mAP50: {float(metrics.box.map50):.3f}")
    return rows

per_class_n = print_map_table(metrics_n)
mAP50_n = round(float(metrics_n.box.map50), 4)

## OPTIONAL second session: yolov8s @ 50 epochs

Commented out by default so free-tier users don't blow the session budget.
Uncomment ONLY if yolov8n mAP is insufficient and you have a fresh session.

In [ ]:
# model_s = YOLO('yolov8s.pt')
# model_s.train(
#     data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'),
#     epochs=50, imgsz=640, batch=12, patience=15,
#     name='india_yolov8s',
# )
# run_dir_s = model_s.trainer.save_dir
# BEST_S = os.path.join(run_dir_s, 'weights', 'best.pt')
# metrics_s = YOLO(BEST_S).val(data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'), verbose=False)
# per_class_s = print_map_table(metrics_s)
# mAP50_s = round(float(metrics_s.box.map50), 4)

BEST_S = None  # set to best.pt path if you ran the s-model above

## Export ONNX (fp32)

Exports `best.pt` to ONNX (imgsz 640, opset 17, simplified) for every trained
size.

In [ ]:
import os as _os
assert 'BEST_N' in dir() and _os.path.isfile(BEST_N), (
    'BEST_N missing: the train cell did not finish in THIS kernel. '
    'If the runtime restarted, /content/runs is gone too — check '
    '!ls /content/runs/detect/. Weights present but variable lost? '
    're-set BEST_N to the best.pt path manually and re-run this cell.')
sizes = {'n': BEST_N}
if BEST_S:
    sizes['s'] = BEST_S

onnx_paths = {}
for tag, pt in sizes.items():
    print(f'exporting yolov8{tag} ...')
    onnx_paths[tag] = YOLO(pt).export(format='onnx', imgsz=640, opset=17, simplify=True)
print(onnx_paths)

## int8 static quantization

Calibrates over `calibration/frames/*.jpg` (preprocess: resize 640x640,
BGR->RGB, /255, NCHW float32). Falls back cleanly to dynamic quantization
(static is preferred) if `quantize_static` fails in Colab.

In [ ]:
import cv2, numpy as np, traceback
from onnxruntime.quantization import CalibrationDataReader, quantize_static, quantize_dynamic, QuantType

class FrameReader(CalibrationDataReader):
    def __init__(self, frames, onnx_path):
        self.input_name = ort_session_input_name(onnx_path)
        self.reiter = iter([self._pre(p) for p in frames])
    def _pre(self, path):
        img = cv2.imread(path)
        assert img is not None, f'cannot read {path}'
        img = cv2.resize(img, (640, 640))  # simple resize is fine for calibration
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        return {self.input_name: np.transpose(rgb, (2, 0, 1))[np.newaxis]}
    def get_next(self):
        return next(self.reiter, None)

def ort_session_input_name(onnx_path):
    import onnxruntime as ort
    return ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider']).get_inputs()[0].name

frames = sorted(glob.glob(os.path.join(DRIVE_DATASET_ROOT, 'calibration', 'frames', '*.jpg')))[:200]
assert frames, 'no calibration frames found'

int8_paths = {}
for tag, onx in onnx_paths.items():
    int8_out = onx.replace('.onnx', '-int8.onnx')
    try:
        reader = FrameReader(frames, onx)
        quantize_static(onx, int8_out, reader, weight_type=QuantType.QInt8)
        print(f'yolov8{tag}: STATIC int8 -> {int8_out}')
    except Exception:
        traceback.print_exc()
        print(f'yolov8{tag}: quantize_static failed, falling back to DYNAMIC '
              '(static is preferred; consider rerunning this cell)')
        quantize_dynamic(onx, int8_out, weight_type=QuantType.QInt8)
    int8_paths[tag] = int8_out
print(int8_paths)

## Promotion gate

Hard bars (continuity with Option A): overall mAP50 ≥ 0.50 **and** `auto` ≥
0.35. Bicycle has a watch floor (0.30) that only prints a note — with 4.7k
vs 171k boxes, failing the run on imbalance alone would waste the session;
re-weighting is a follow-up experiment on evidence.

In [ ]:
GATE_OVERALL = 0.50
GATE_AUTO = 0.35
WATCH_BICYCLE = 0.30  # watch-item only (4.7k boxes vs 171k two-wheeler)

auto_map = per_class_n.get("auto")
print("gate: overall mAP50=%.3f (bar %s), auto mAP50=%s (bar %s)" % (mAP50_n, GATE_OVERALL, auto_map, GATE_AUTO))
print("per-class:", {k: round(v, 3) for k, v in per_class_n.items()})
assert mAP50_n >= GATE_OVERALL, "overall mAP50 below bar"
assert auto_map is not None and auto_map >= GATE_AUTO, "auto mAP50 below bar"
bic = per_class_n.get("bicycle")
if bic is not None and bic < WATCH_BICYCLE:
    print("WATCH: bicycle mAP50=%.3f below %s - imbalance note, not a failure" % (bic, WATCH_BICYCLE))
print("gate PASSED: packaging registry artifacts")


## Package model registry

Creates `models/registry/india-yolov8{n,s}/` containing the ONNX models and
`metadata.json` matching the repo contract exactly.

In [ ]:
import datetime, json

registry_root = 'models/registry'
os.makedirs(registry_root, exist_ok=True)

def build_meta(name, quant, source_run, map50, per_class):
    return {
        'name': name,
        'classes': CLASSES,
        'imgsz': 640,
        'normalization': {'mean': [0, 0, 0], 'std': [255, 255, 255],
                          'layout': 'NCHW', 'color': 'RGB'},
        'quantization': quant,
        'source_run': source_run,
        'metrics': {'mAP50_overall': map50, 'per_class_mAP50': per_class},
        'provenance': {'hf_repo': HF_REPO,  # iisc-aim/BMD-45 official 'hf_subset': HF_SUBSET,
                       'mapping': 'optionB-merge-6class',
                       'contract': 'docs/DATASET_SPEC.md v1',
                       'val': 'official-BMD-45-Val',
                       'epochs': EPOCHS[tag]},
    }

import shutil

EPOCHS = {'n': 30}
if BEST_S:
    EPOCHS['s'] = 50
source_run = datetime.date.today().isoformat()

# One metadata.json per size describes the deployed (int8) artifact;
# model.onnx stays alongside for fp32 inference.
for tag, onx in onnx_paths.items():
    d = os.path.join(registry_root, f'india-yolov8{tag}')
    os.makedirs(d, exist_ok=True)
    shutil.copy(onx, os.path.join(d, 'model.onnx'))
    shutil.copy(int8_paths[tag], os.path.join(d, 'model-int8.onnx'))
    map50 = mAP50_n if tag == 'n' else mAP50_s
    per_cls = per_class_n if tag == 'n' else per_class_s
    meta = build_meta(f'india-yolov8{tag}', 'int8', source_run, map50, per_cls)
    with open(os.path.join(d, 'metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)

for root_d, _, fs in os.walk(registry_root):
    for fn in fs:
        print(os.path.join(root_d, fn))

## Zip registry and download

Zips `models/registry/` and downloads it via the browser — the only artifact that leaves Colab.

In [ ]:
import shutil

zip_base = 'india_model_registry'
shutil.make_archive(zip_base, 'zip', root_dir='.', base_dir='models/registry')
zip_path = zip_base + '.zip'
print('created:', zip_path)

from google.colab import files
files.download(zip_path)  # <- the ONLY artifact that leaves Colab

print()
print('Local handoff: unzip so models/registry/india-yolov8*/ lands at repo root,')
print('then run make verify + make eval and update MASTER_PLAN Phase T.')

## Handoff checklist

1. The browser downloads `india_model_registry.zip` (previous cell).
2. Unzip into the repo so `models/registry/india-yolov8n/` lands at root.
3. `metadata.json` carries `mapping: optionB-merge-6class` + HF provenance.
4. Run `make verify` + `make eval`; update MASTER_PLAN.
5. If the bicycle WATCH fires: follow-up experiment is loss re-weighting,
   not a failed run.
